# Run the pipeline on Colab

One environment per run. Stages: `references` (LeWM and full-grid DINO-WM on the pinned queries),
`features` (cache pooled DINOv2 tokens), `train` (one predictor per token budget), `evaluate`, `plot`.
Datasets, features, checkpoints and results are mirrored to Google Drive, so stages can run in separate
sessions and training resumes after a disconnect. Use an A100 runtime for `train` and for any timing row
that goes into the same table.

In [ ]:
ENV = "pusht"                                    # tworoom | pusht | reacher | cube
STAGES = "references,features,train,evaluate,plot"
EXTRA = "--episodes 2000 --steps 30000"          # see `dwt run --help`
REPO = "https://github.com/tjayada/dino-wm-tokens.git"

In [ ]:
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
if not Path("/content/dino-wm-tokens").exists():
    subprocess.run(["git", "clone", "-q", REPO, "/content/dino-wm-tokens"], check=True)
os.chdir("/content/dino-wm-tokens")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=True)
subprocess.run(["bash", "scripts/colab_setup.sh"], check=True)

os.environ["STABLEWM_HOME"] = "/content/stablewm"                       # session disk, fast
os.environ["DWT_PERSIST"] = "/content/drive/MyDrive/dino-wm-tokens"     # survives the session
os.environ["DWT_RESULTS"] = "/content/drive/MyDrive/dino-wm-tokens/results"

In [ ]:
!dwt run --env $ENV --stages $STAGES $EXTRA

Results land in `MyDrive/dino-wm-tokens/results/` (`runs.csv`, queries, training curves, figures);
copy them into the repository's `results/` folder to commit them.